# Gradio Demo - So sánh ViT5 Stage 2 và Stage 3 PPO

Notebook này tạo giao diện Gradio để:
- Chọn văn bản có sẵn từ `test/dataset.jsonl` trên Google Drive.
- Hoặc tự dán văn bản đầu vào.
- Sinh tóm tắt bằng hai mô hình: Stage 2 SFT và Stage 3 PPO.
- Hiển thị hai output riêng biệt và các chỉ số so sánh với bản tóm tắt tham chiếu nếu có.

Ghi chú: Nếu người dùng tự dán văn bản không có `gold summary`, hệ thống vẫn tính được các chỉ số dựa trên source như `number_hallucination`, `length_ratio`, `length_compliance`, nhưng không tính ROUGE/BERTScore.

In [ ]:
# Cell 1 - Install dependencies if needed
import sys
import subprocess
import importlib.metadata as md

INSTALL_DEPS = True

PACKAGES = [
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "accelerate==1.1.1",
    "evaluate==0.4.3",
    "rouge-score==0.1.2",
    "bert-score==0.3.13",
    "sentencepiece==0.2.0",
    "gradio==5.9.1",
]

missing = []
for spec in PACKAGES:
    name, target = spec.split("==")
    dist_name = "rouge-score" if name == "rouge-score" else name
    try:
        current = md.version(dist_name)
    except md.PackageNotFoundError:
        missing.append(spec)
        continue
    if current != target:
        missing.append(spec)

if missing and INSTALL_DEPS:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade", *missing])
    raise RuntimeError("Installation finished. Restart runtime, then run from Cell 2.")
elif missing:
    print("Missing/version mismatch:", missing)
else:
    print("Environment OK.")

In [ ]:
# Cell 2 - Imports and Drive mount
import os
import re
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import gradio as gr
import evaluate
from bert_score import score as bertscore
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig, T5Tokenizer

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print('Skip Colab Drive mount:', repr(e))

# Patch evaluate/HuggingFace token lookup in some Colab environments.
try:
    import huggingface_hub.hf_api
    class _MockHfFolder:
        @classmethod
        def get_token(cls):
            return None
    huggingface_hub.hf_api.HfFolder = _MockHfFolder
except Exception as e:
    print('Skip HfFolder patch:', repr(e))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
print('Device:', DEVICE)
print('Precision:', 'bf16' if USE_BF16 else 'fp32')

In [ ]:
# Cell 3 - Configuration

# Data path: mac dinh dung test set de so sanh cong bang Stage 2 vs Stage 3.
# Notebook ho tro ca cau truc MyNLPData/test va MyNLPData/data/test.
DATA_CANDIDATES = [
    '/content/drive/MyDrive/MyNLPData/test/dataset.jsonl',
    '/content/drive/MyDrive/MyNLPData/data/test/dataset.jsonl',
]

# Stage 3 notebook KHONG load Stage 2 tu best_model.
# No load checkpoint-735, sau do repair sang BASE_DIR_S3/stage2_repaired_for_stage3 neu thieu config/tokenizer.
STAGE2_RAW_DIR = (
    '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP'
    '/ViT5_stage_2/runs/vit5_base_stage2/checkpoints/checkpoint-735'
)

STAGE2_REPAIRED_DIR = (
    '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP'
    '/ViT5_stage_3/stage2_repaired_for_stage3'
)

STAGE3_MODEL_DIR = (
    '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP'
    '/ViT5_stage_3/runs/ppo_run/best_model'
)

GEN_CONFIG = {
    'MAX_INPUT_LEN': 1024,
    'GEN_MAX_LEN': 320,
    'GEN_MIN_LEN': 64,
    'NUM_BEAMS': 4,
    'NO_REPEAT_NGRAM': 3,
    'LENGTH_PENALTY': 1.0,
    'MIN_RATIO': 0.18,
    'MAX_RATIO': 0.27,
}

for p in DATA_CANDIDATES:
    if Path(p).exists():
        DATA_PATH = Path(p)
        break
else:
    raise FileNotFoundError('Khong tim thay test/dataset.jsonl trong DATA_CANDIDATES. Hay sua DATA_CANDIDATES trong Cell 3.')

if not Path(STAGE2_RAW_DIR).exists() and not Path(STAGE2_REPAIRED_DIR).exists():
    raise FileNotFoundError(
        'Khong tim thay Stage 2 checkpoint. Da thu:\n'
        f' - raw: {STAGE2_RAW_DIR}\n'
        f' - repaired: {STAGE2_REPAIRED_DIR}'
    )

if not Path(STAGE3_MODEL_DIR).exists():
    raise FileNotFoundError(f'Khong tim thay Stage 3 model: {STAGE3_MODEL_DIR}')

print('Data:', DATA_PATH)
print('Stage 2 raw:', STAGE2_RAW_DIR)
print('Stage 2 repaired:', STAGE2_REPAIRED_DIR)
print('Stage 3:', STAGE3_MODEL_DIR)

In [ ]:
# Cell 4 - Load dataset samples

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalize_record(row, idx):
    source = str(row.get('source', '')).strip()
    summary = str(row.get('summary', '')).strip()
    title = str(row.get('title', '') or row.get('id', '') or f'Mau {idx}').strip()
    category = str(row.get('category', '')).strip()
    src_words = len(source.split())
    prefix = f'{idx:04d}'
    if category:
        prefix += f' | {category}'
    label = f'{prefix} | {src_words} tu | {title[:90]}'
    return {
        'idx': idx,
        'label': label,
        'source': source,
        'summary': summary,
        'category': category,
        'id': row.get('id', ''),
    }

raw_rows = load_jsonl(DATA_PATH)
SAMPLES = [normalize_record(r, i) for i, r in enumerate(raw_rows) if str(r.get('source', '')).strip()]

# Optional: load exported predictions to filter samples where Stage 2 and Stage 3 differ.
# Neu cac file nay chua ton tai, filter van hoat dong voi cac tieu chi dua tren source/gold.
COMBINED_PREDICTION_CANDIDATES = [
    '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/results/stage2_stage3_test_predictions.jsonl',
]

PREDICTION_CANDIDATES = {
    'stage2': [
        '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2/runs/vit5_base_stage2/results/stage2_predictions.jsonl',
    ],
    'stage3': [
        '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_3/runs/ppo_run/results/stage3_predictions.jsonl',
    ],
}

NUM_RE_LOCAL = re.compile(r'\d+(?:[.,]\d+)*')

def _nums(text):
    return set(NUM_RE_LOCAL.findall(text or ''))

def _num_hallucination(src, pred):
    pred_nums = _nums(pred)
    if not pred_nums:
        return 0.0
    return len(pred_nums - _nums(src)) / len(pred_nums)

def _norm_text(text):
    return ' '.join(str(text or '').split()).lower()

def _first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

def _load_prediction_rows(paths):
    p = _first_existing(paths)
    if p is None:
        return [], None
    return load_jsonl(p), p

combined_pred_rows, combined_pred_path = _load_prediction_rows(COMBINED_PREDICTION_CANDIDATES)
stage2_pred_rows, stage2_pred_path = _load_prediction_rows(PREDICTION_CANDIDATES['stage2'])
stage3_pred_rows, stage3_pred_path = _load_prediction_rows(PREDICTION_CANDIDATES['stage3'])

def _index_predictions(rows, pred_keys):
    by_id = {}
    by_idx = {}
    for i, row in enumerate(rows):
        pred = ''
        for key in pred_keys:
            if row.get(key):
                pred = str(row.get(key, '')).strip()
                break
        item = {'pred': pred, 'row': row}
        if row.get('id'):
            by_id[str(row.get('id'))] = item
        by_idx[i] = item
    return by_id, by_idx

combined_s2_by_id, combined_s2_by_idx = _index_predictions(combined_pred_rows, ['pred_stage2', 'pred_sft'])
combined_s3_by_id, combined_s3_by_idx = _index_predictions(combined_pred_rows, ['pred_stage3', 'pred_ppo'])
s2_by_id, s2_by_idx = _index_predictions(stage2_pred_rows, ['pred_sft', 'pred_stage2', 'prediction'])
s3_by_id, s3_by_idx = _index_predictions(stage3_pred_rows, ['pred_ppo', 'pred_stage3', 'prediction'])

for sample in SAMPLES:
    sid = str(sample.get('id', ''))
    idx = sample['idx']
    s2 = combined_s2_by_id.get(sid) or combined_s2_by_idx.get(idx) or s2_by_id.get(sid) or s2_by_idx.get(idx)
    s3 = combined_s3_by_id.get(sid) or combined_s3_by_idx.get(idx) or s3_by_id.get(sid) or s3_by_idx.get(idx)
    sample['pred_stage2_exported'] = s2['pred'] if s2 else ''
    sample['pred_stage3_exported'] = s3['pred'] if s3 else ''
    sample['source_num_count'] = len(_nums(sample['source']))
    sample['gold_num_count'] = len(_nums(sample['summary']))
    sample['exported_diff'] = bool(
        sample['pred_stage2_exported'] and sample['pred_stage3_exported']
        and _norm_text(sample['pred_stage2_exported']) != _norm_text(sample['pred_stage3_exported'])
    )
    sample['stage2_exported_halluc'] = _num_hallucination(sample['source'], sample['pred_stage2_exported']) if sample['pred_stage2_exported'] else None
    sample['stage3_exported_halluc'] = _num_hallucination(sample['source'], sample['pred_stage3_exported']) if sample['pred_stage3_exported'] else None

DIFF_SAMPLES = [r for r in SAMPLES if r['exported_diff']]
if not DIFF_SAMPLES:
    raise RuntimeError(
        'Không có sample khác nhau giữa Stage 2 và Stage 3 để hiển thị. '
        'Hãy chạy notebook 01_generate_stage2_stage3_test_predictions.ipynb trước để tạo file predictions.'
    )

FILTER_OPTIONS = [
    'Tất cả mẫu khác nhau',
    'Khác nhau + nhiều số trong source (>=3)',
    'Khác nhau + có số trong gold summary',
    'Khác nhau + Stage 2 có hallucination số',
    'Khác nhau + Stage 3 giảm hallucination so với Stage 2',
]

def choices_for_filter(filter_name):
    rows = DIFF_SAMPLES
    if filter_name == 'Khác nhau + nhiều số trong source (>=3)':
        rows = [r for r in rows if r['source_num_count'] >= 3]
    elif filter_name == 'Khác nhau + có số trong gold summary':
        rows = [r for r in rows if r['gold_num_count'] > 0]
    elif filter_name == 'Khác nhau + Stage 2 có hallucination số':
        rows = [r for r in rows if r['stage2_exported_halluc'] is not None and r['stage2_exported_halluc'] > 0]
    elif filter_name == 'Khác nhau + Stage 3 giảm hallucination so với Stage 2':
        rows = [r for r in rows if r['stage2_exported_halluc'] is not None and r['stage3_exported_halluc'] is not None and r['stage3_exported_halluc'] < r['stage2_exported_halluc']]
    return [r['label'] for r in rows]

CHOICES = choices_for_filter(FILTER_OPTIONS[0])
LABEL_TO_SAMPLE = {r['label']: r for r in SAMPLES}

print('Loaded samples:', len(SAMPLES))
print('Combined predictions:', combined_pred_path or 'not found')
print('Stage 2 predictions fallback:', stage2_pred_path or 'not found')
print('Stage 3 predictions fallback:', stage3_pred_path or 'not found')
print('Different exported predictions:', len(DIFF_SAMPLES))
print('First sample:', CHOICES[0] if CHOICES else 'EMPTY')

In [ ]:
# Cell 5 - Load Stage 2 and Stage 3 models

def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

STANDARD_WEIGHT_NAMES = {
    'model.safetensors',
    'pytorch_model.bin',
    'tf_model.h5',
    'flax_model.msgpack',
}

def has_required_hf_files(path):
    path = Path(path)
    if path.is_file():
        return False
    has_config = (path / 'config.json').exists()
    has_tokenizer = any((path / name).exists() for name in ('tokenizer_config.json', 'spiece.model', 'tokenizer.json'))
    has_weight = any((path / name).exists() for name in STANDARD_WEIGHT_NAMES)
    return has_config and has_tokenizer and has_weight

def find_stage2_weight_files(path):
    path = Path(path)
    exact = [path / name for name in STANDARD_WEIGHT_NAMES if (path / name).exists()]
    if exact:
        return exact
    candidates = []
    for pattern in ('*.safetensors', '*.bin', '*.pt', '*.pth', '*.tensors'):
        candidates.extend(path.glob(pattern))
    candidates = [x for x in candidates if x.is_file()]
    if not candidates:
        raise FileNotFoundError(f'Khong thay weight file trong {path}')
    return [max(candidates, key=lambda x: x.stat().st_size)]

def infer_standard_weight_name(src):
    lower = Path(src).name.lower()
    if lower == 'model.safetensors' or lower.endswith('.safetensors'):
        return 'model.safetensors'
    if lower == 'pytorch_model.bin' or lower.endswith(('.bin', '.pt', '.pth')):
        return 'pytorch_model.bin'
    if lower.endswith('.tensors'):
        try:
            from safetensors import safe_open
            with safe_open(str(src), framework='pt', device='cpu') as f:
                _ = list(f.keys())[:1]
            return 'model.safetensors'
        except Exception:
            return 'pytorch_model.bin'
    return Path(src).name

def repair_stage2_checkpoint(raw_dir, repaired_dir):
    raw_dir = Path(raw_dir)
    repaired_dir = Path(repaired_dir)
    if has_required_hf_files(repaired_dir):
        print('Dung Stage 2 repaired checkpoint:', repaired_dir)
        return str(repaired_dir)
    if has_required_hf_files(raw_dir):
        print('Stage 2 raw checkpoint da du file HF:', raw_dir)
        return str(raw_dir)

    repaired_dir.mkdir(parents=True, exist_ok=True)
    cfg = AutoConfig.from_pretrained('VietAI/vit5-base')
    cfg.save_pretrained(repaired_dir)
    tok = T5Tokenizer.from_pretrained('VietAI/vit5-base', legacy=True)
    tok.save_pretrained(repaired_dir)

    copied = []
    import shutil
    for src in find_stage2_weight_files(raw_dir):
        dst = repaired_dir / infer_standard_weight_name(src)
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
        copied.append(dst.name)
    print('Da tao Stage 2 repaired checkpoint:', repaired_dir)
    print('Weight files:', copied)
    return str(repaired_dir)

def load_model_and_tokenizer(model_dir):
    tok = AutoTokenizer.from_pretrained(model_dir, use_fast=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_dir,
        torch_dtype=MODEL_DTYPE,
    ).to(DEVICE)
    model.eval()
    return tok, model

clear_cuda()
STAGE2_MODEL_DIR = repair_stage2_checkpoint(STAGE2_RAW_DIR, STAGE2_REPAIRED_DIR)
stage2_tokenizer, stage2_model = load_model_and_tokenizer(STAGE2_MODEL_DIR)
stage3_tokenizer, stage3_model = load_model_and_tokenizer(STAGE3_MODEL_DIR)
rouge = evaluate.load('rouge')

print('Stage 2 params:', round(sum(p.numel() for p in stage2_model.parameters()) / 1e6, 1), 'M')
print('Stage 3 params:', round(sum(p.numel() for p in stage3_model.parameters()) / 1e6, 1), 'M')

In [ ]:
# Cell 6 - Generation and metrics

NUM_RE = re.compile(r'\d+(?:[.,]\d+)*')

def extract_numbers(text):
    return set(NUM_RE.findall(text or ''))

def normalize_for_rouge(text):
    return '\n'.join(str(text).strip().split())

def generate_one(model, tok, text):
    enc = tok(
        [text],
        max_length=GEN_CONFIG['MAX_INPUT_LEN'],
        truncation=True,
        padding=True,
        return_tensors='pt',
    ).to(DEVICE)
    with torch.no_grad():
        ids = model.generate(
            **enc,
            max_new_tokens=GEN_CONFIG['GEN_MAX_LEN'],
            min_new_tokens=GEN_CONFIG['GEN_MIN_LEN'],
            num_beams=GEN_CONFIG['NUM_BEAMS'],
            no_repeat_ngram_size=GEN_CONFIG['NO_REPEAT_NGRAM'],
            length_penalty=GEN_CONFIG['LENGTH_PENALTY'],
            early_stopping=True,
        )
    return tok.batch_decode(ids, skip_special_tokens=True)[0].strip()

def number_recall_against_gold(gold, pred):
    gold_nums = extract_numbers(gold)
    if not gold_nums:
        return np.nan
    return len(gold_nums & extract_numbers(pred)) / len(gold_nums)

def number_recall_against_source(src, pred):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(pred)) / len(src_nums)

def number_hallucination(src, pred):
    pred_nums = extract_numbers(pred)
    if not pred_nums:
        return 0.0
    return len(pred_nums - extract_numbers(src)) / len(pred_nums)

def length_ratio(src, pred):
    return len(str(pred).split()) / max(len(str(src).split()), 1)

def length_compliance(src, pred):
    ratio = length_ratio(src, pred)
    return float(GEN_CONFIG['MIN_RATIO'] <= ratio <= GEN_CONFIG['MAX_RATIO'])

def compute_metrics_for_output(src, gold, pred):
    metrics = {
        'source_number_recall': round(number_recall_against_source(src, pred) * 100, 4),
        'number_hallucination': round(number_hallucination(src, pred) * 100, 4),
        'length_compliance': round(length_compliance(src, pred) * 100, 4),
        'length_ratio': round(length_ratio(src, pred), 4),
        'output_words': len(pred.split()),
    }
    if gold and gold.strip():
        rouge_result = rouge.compute(
            predictions=[normalize_for_rouge(pred)],
            references=[normalize_for_rouge(gold)],
            use_stemmer=False,
        )
        _, _, bert_f1 = bertscore(
            [pred],
            [gold],
            lang='vi',
            model_type='xlm-roberta-base',
            verbose=False,
        )
        metrics.update({
            'rouge1': round(rouge_result['rouge1'] * 100, 4),
            'rouge2': round(rouge_result['rouge2'] * 100, 4),
            'rougeL': round(rouge_result['rougeL'] * 100, 4),
            'bertscore_f1': round(float(bert_f1.mean().item()) * 100, 4),
            'gold_number_recall': round(number_recall_against_gold(gold, pred) * 100, 4) if not np.isnan(number_recall_against_gold(gold, pred)) else None,
        })
    else:
        metrics.update({
            'rouge1': None,
            'rouge2': None,
            'rougeL': None,
            'bertscore_f1': None,
            'gold_number_recall': None,
        })
    return metrics

def compare_models(src, gold):
    pred_s2 = generate_one(stage2_model, stage2_tokenizer, src)
    pred_s3 = generate_one(stage3_model, stage3_tokenizer, src)
    m2 = compute_metrics_for_output(src, gold, pred_s2)
    m3 = compute_metrics_for_output(src, gold, pred_s3)
    df = pd.DataFrame([
        {'model': 'Stage 2 SFT', **m2},
        {'model': 'Stage 3 PPO', **m3},
    ])
    return pred_s2, pred_s3, df

In [ ]:
# Cell 7 - Gradio application

def filter_samples(filter_name):
    choices = choices_for_filter(filter_name)
    value = choices[0] if choices else None
    status = f'Filter: {filter_name} | {len(choices)} mẫu'
    if not choices:
        status += ' | Không có mẫu phù hợp hoặc thiếu file predictions.'
    return gr.update(choices=choices, value=value), status

def load_selected_sample(choice):
    if not choice or choice not in LABEL_TO_SAMPLE:
        return '', '', ''
    row = LABEL_TO_SAMPLE[choice]
    extra = []
    if row.get('source_num_count') is not None:
        extra.append(f"source_nums={row['source_num_count']}")
    if row.get('gold_num_count') is not None:
        extra.append(f"gold_nums={row['gold_num_count']}")
    if row.get('exported_diff'):
        extra.append('exported_stage2_stage3=different')
    if row.get('stage2_exported_halluc') is not None:
        extra.append(f"s2_halluc={row['stage2_exported_halluc']:.3f}")
    if row.get('stage3_exported_halluc') is not None:
        extra.append(f"s3_halluc={row['stage3_exported_halluc']:.3f}")
    info = f"id={row.get('id', '')} | category={row.get('category', '')}"
    if extra:
        info += ' | ' + ' | '.join(extra)
    return row['source'], row['summary'], info

def run_demo(choice, custom_text, use_custom_text):
    if use_custom_text and custom_text.strip():
        src = custom_text.strip()
        gold = ''
        sample_info = 'Custom input: khong co gold summary, chi tinh metric dua tren source.'
    else:
        if not choice or choice not in LABEL_TO_SAMPLE:
            return '', '', '', pd.DataFrame(), 'Hay chon mot mau co san hoac nhap custom text.'
        row = LABEL_TO_SAMPLE[choice]
        src = row['source']
        gold = row['summary']
        sample_info = f"Dataset sample | id={row.get('id', '')} | category={row.get('category', '')}"

    if not src.strip():
        return '', '', '', pd.DataFrame(), 'Input rong.'

    pred_s2, pred_s3, metrics_df = compare_models(src, gold)
    return pred_s2, pred_s3, gold, metrics_df, sample_info

with gr.Blocks(title='ViT5 Stage 2 vs Stage 3 PPO') as demo:
    gr.Markdown('# So sánh ViT5 Stage 2 SFT và Stage 3 PPO')
    gr.Markdown('Chọn một văn bản từ tập test hoặc tự dán văn bản. Nhấn Generate để sinh tóm tắt bằng cả hai mô hình.')

    with gr.Row():
        sample_filter = gr.Dropdown(
            choices=FILTER_OPTIONS,
            value=FILTER_OPTIONS[0],
            label='Lọc mẫu test',
        )
        sample_dropdown = gr.Dropdown(
            choices=CHOICES,
            value=CHOICES[0] if CHOICES else None,
            label='Văn bản có sẵn từ test/dataset.jsonl',
            filterable=True,
        )
        use_custom = gr.Checkbox(label='Dùng văn bản tự dán thay vì mẫu có sẵn', value=False)

    filter_status = gr.Textbox(label='Trạng thái filter', value=f'Filter: {FILTER_OPTIONS[0]} | {len(CHOICES)} mẫu', interactive=False)
    source_box = gr.Textbox(label='Source document', lines=12)
    custom_box = gr.Textbox(label='Custom input', lines=8, placeholder='Dán văn bản y tế cần tóm tắt tại đây...')
    sample_info = gr.Textbox(label='Thông tin mẫu', interactive=False)

    load_btn = gr.Button('Load selected sample')
    generate_btn = gr.Button('Generate', variant='primary')

    with gr.Row():
        stage2_output = gr.Textbox(label='Output Stage 2 SFT', lines=10)
        stage3_output = gr.Textbox(label='Output Stage 3 PPO', lines=10)

    gold_output = gr.Textbox(label='Gold summary trong dataset nếu có', lines=6)
    metrics_table = gr.Dataframe(label='Chỉ số so sánh', wrap=True)

    sample_filter.change(
        fn=filter_samples,
        inputs=[sample_filter],
        outputs=[sample_dropdown, filter_status],
    )

    load_btn.click(
        fn=load_selected_sample,
        inputs=[sample_dropdown],
        outputs=[source_box, gold_output, sample_info],
    )

    generate_btn.click(
        fn=run_demo,
        inputs=[sample_dropdown, custom_box, use_custom],
        outputs=[stage2_output, stage3_output, gold_output, metrics_table, sample_info],
    )

demo.launch(share=True, debug=True)